In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import pymysql
import json
import os
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# ======================
# 配置参数
# ======================
N_TRIALS = 5  # 训练次数
TEACHER_EPOCHS = 50
STUDENT_EPOCHS = 50
LEARNING_RATE = 0.001
OUTPUT_DIR = "best_model"

# 创建输出目录
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ======================
# 数据库连接
# ======================
def get_df(db, sql):
    conn = pymysql.connect(
        host="localhost",
        user="root",
        password="123456",
        database=db,
        charset="utf8mb4"
    )
    df = pd.read_sql(sql, conn)
    conn.close()
    return df

# 检查连接
try:
    get_df("yx_ai_feature", "SELECT 1")
    print("✓ 数据库连接成功！")
except Exception as e:
    print(f"✗ 数据库连接失败: {e}")
    exit()

# 读取数据（包含 ip_count）
df = get_df("yx_ai_feature", """
    SELECT scan_count, time_variance, location_variance, device_count, ip_count, is_reused
    FROM reuse_pattern
""")
print(f"✓ 读取数据: {len(df)} 条记录")
print(f"✓ 类别分布: {df['is_reused'].value_counts().to_dict()}")

# ======================
# 缺失值处理
# ======================
print("\n✓ 检查缺失值:")
missing_counts = df.isnull().sum()
print(missing_counts)

original_len = len(df)
df = df.dropna(subset=["is_reused"])
if len(df) < original_len:
    print(f"\n✓ 删除标签缺失样本: {original_len - len(df)} 条")

FEATURES = ["scan_count", "time_variance", "location_variance", "device_count", "ip_count"]
for col in FEATURES:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"✓ 填充 {col} 缺失值，中位数: {median_val:.2f}")

print(f"\n✓ 处理后样本数: {len(df)}")

# ======================
# 数据预处理
# ======================
X = df[FEATURES].values
y = df["is_reused"].values

# 计算类别权重（解决数据不平衡）
class_counts = np.bincount(y)
class_weights = torch.tensor([class_counts[0]/len(y), class_counts[1]/len(y)], dtype=torch.float32)
print(f"\n✓ 类别权重: 负样本={class_weights[0]:.4f}, 正样本={class_weights[1]:.4f}")

# 标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 打印标准化参数
print(f"\n✓ 标准化参数:")
print(f"  特征顺序: {FEATURES}")
print(f"  均值 (means): {scaler.mean_.tolist()}")
print(f"  标准差 (stds): {scaler.scale_.tolist()}")

# 分层划分训练测试集（保持类别比例）
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

# 转 PyTorch 张量
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)

# ======================
# 教师模型（增强版双通道）
# ======================
class TeacherModel(nn.Module):
    def __init__(self, in_dim=5, hidden_dim=32):
        super().__init__()
        self.time_proj = nn.Linear(in_dim, hidden_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=hidden_dim * 2,
            batch_first=True,
            dropout=0.1
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.graph_branch = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.BatchNorm1d(hidden_dim)
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, x):
        seq = self.time_proj(x).unsqueeze(1)
        seq_out = self.transformer(seq).squeeze(1)
        graph_out = self.graph_branch(x)
        fused = torch.cat([seq_out, graph_out], dim=1)
        return self.head(fused)

# ======================
# 学生模型（优化版）
# ======================
class StudentModel(nn.Module):
    def __init__(self, in_dim=5):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.2),
            nn.Linear(16, 2)
        )

    def forward(self, x):
        return self.fc(x)

# ======================
# 评估函数
# ======================
def evaluate(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_prob) if y_prob is not None else "N/A"
    cm = confusion_matrix(y_true, y_pred)
    return acc, prec, recall, f1, auc, cm

# ======================
# 单次训练函数
# ======================
def train_single_trial(trial_num):
    print(f"\n" + "="*60)
    print(f"          第 {trial_num} 次训练")
    print("="*60)

    # 设置随机种子（保证可重复性）
    torch.manual_seed(trial_num * 42)
    np.random.seed(trial_num * 42)

    # 训练教师模型
    print("\n--- 训练教师模型 ---")
    teacher = TeacherModel()
    opt = torch.optim.AdamW(teacher.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TEACHER_EPOCHS)

    for e in range(TEACHER_EPOCHS):
        teacher.train()
        logits = teacher(X_train)
        loss = criterion(logits, y_train)
        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()

        if (e+1) % max(1, TEACHER_EPOCHS // 5) == 0:
            teacher.eval()
            with torch.no_grad():
                val_logits = teacher(X_test)
                val_acc = (torch.argmax(val_logits, dim=1) == y_test).float().mean().item()
            print(f"第{e+1}轮 | 损失: {loss.item():.4f} | 验证准确率: {val_acc:.4f}")

    # 知识蒸馏训练学生模型
    print("\n--- 知识蒸馏训练学生模型 ---")
    student = StudentModel()
    opt = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=STUDENT_EPOCHS)
    ce_criterion = nn.CrossEntropyLoss(weight=class_weights)
    kl_criterion = nn.KLDivLoss(reduction="batchmean")

    for e in range(STUDENT_EPOCHS):
        student.train()
        teacher.eval()

        with torch.no_grad():
            t_out = teacher(X_train)
        s_out = student(X_train)

        ce_loss = ce_criterion(s_out, y_train)
        kl_loss = kl_criterion(F.log_softmax(s_out / 2, dim=1), F.softmax(t_out / 2, dim=1)) * (2.0 ** 2)
        loss = 0.6 * ce_loss + 0.4 * kl_loss

        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()

        if (e+1) % max(1, STUDENT_EPOCHS // 5) == 0:
            student.eval()
            with torch.no_grad():
                val_logits = student(X_test)
                val_acc = (torch.argmax(val_logits, dim=1) == y_test).float().mean().item()
            print(f"第{e+1}轮 | 损失: {loss.item():.4f} | 验证准确率: {val_acc:.4f}")

    # 评估学生模型
    student.eval()
    with torch.no_grad():
        test_logits = student(X_test)
        test_pred = torch.argmax(test_logits, dim=1)
        test_proba = torch.softmax(test_logits, dim=1)[:, 1]

    test_acc, test_prec, test_recall, test_f1, test_auc, cm = evaluate(
        y_test.numpy(), test_pred.numpy(), test_proba.numpy()
    )

    print(f"\n【第{trial_num}次训练结果】")
    print(f"  准确率: {test_acc:.4f}")
    print(f"  精确率: {test_prec:.4f}")
    print(f"  召回率: {test_recall:.4f}")
    print(f"  F1分数: {test_f1:.4f}")
    print(f"  AUC: {test_auc:.4f}")

    return student, test_f1, test_auc, test_recall

# ======================
# 多次训练并选择最佳模型
# ======================
print("\n" + "="*60)
print(f"       开始 {N_TRIALS} 次训练，选择最佳模型")
print("="*60)

best_model = None
best_f1 = 0.0
best_auc = 0.0
best_recall = 0.0
best_trial = 0
trial_results = []

for trial in range(1, N_TRIALS + 1):
    student, f1, auc, recall = train_single_trial(trial)
    
    # 记录结果
    trial_results.append({
        "trial": trial,
        "f1": f1,
        "auc": auc,
        "recall": recall
    })

    # 更新最佳模型
    if f1 > best_f1:
        best_f1 = f1
        best_auc = auc
        best_recall = recall
        best_model = student
        best_trial = trial
        print(f"\n✓ 第 {trial} 次训练成为当前最佳！")

# ======================
# 显示所有训练结果
# ======================
print("\n" + "="*60)
print("          所有训练结果汇总")
print("="*60)

print(f"\n{'训练次数':<10} {'F1分数':<10} {'AUC':<10} {'召回率':<10}")
print("-" * 40)
for result in trial_results:
    marker = "★" if result["trial"] == best_trial else ""
    print(f"{result['trial']:<10} {result['f1']:<10.4f} {result['auc']:<10.4f} {result['recall']:<10.4f} {marker}")

print(f"\n✓ 最佳模型来自第 {best_trial} 次训练")
print(f"  F1分数: {best_f1:.4f}")
print(f"  AUC: {best_auc:.4f}")
print(f"  召回率: {best_recall:.4f}")

# ======================
# 阈值调优（针对最佳模型）
# ======================
print("\n" + "="*60)
print("         最佳模型阈值调优")
print("="*60)

best_model.eval()
with torch.no_grad():
    test_logits = best_model(X_test)
    test_proba = torch.softmax(test_logits, dim=1)[:, 1]

best_threshold_f1 = 0
best_threshold = 0.5
for threshold in np.arange(0.1, 0.9, 0.05):
    threshold_pred = (test_proba.numpy() >= threshold).astype(int)
    f1 = f1_score(y_test.numpy(), threshold_pred)
    if f1 > best_threshold_f1:
        best_threshold_f1 = f1
        best_threshold = threshold

print(f"✓ 最佳阈值: {best_threshold:.2f}，对应F1: {best_threshold_f1:.4f}")

# ======================
# 导出最佳模型和配置
# ======================
print("\n" + "="*60)
print("          导出最佳模型")
print("="*60)

# 导出 ONNX 模型
model_path = os.path.join(OUTPUT_DIR, "qs_risk_model.onnx")
torch.onnx.export(
    best_model,
    torch.randn(1, 5),
    model_path,
    input_names=["features"],
    output_names=["logits"],
    dynamic_axes={"features": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=13,
    export_params=True,
    do_constant_folding=True
)

# 导出标准化配置
scaler_config = {
    "version": "1.0",
    "scalerType": "StandardScaler",
    "featureNames": FEATURES,
    "means": scaler.mean_.tolist(),
    "stds": scaler.scale_.tolist(),
    "minValues": None,
    "maxValues": None,
    "updateTime": pd.Timestamp.now().isoformat()
}

scaler_path = os.path.join(OUTPUT_DIR, "qs_risk_scaler.json")
with open(scaler_path, "w") as f:
    json.dump(scaler_config, f, indent=2)

# 导出训练配置
train_config = {
    "modelVersion": "1.0",
    "bestTrial": best_trial,
    "totalTrials": N_TRIALS,
    "metrics": {
        "f1": best_f1,
        "auc": best_auc,
        "recall": best_recall
    },
    "bestThreshold": best_threshold,
    "trainingParams": {
        "teacherEpochs": TEACHER_EPOCHS,
        "studentEpochs": STUDENT_EPOCHS,
        "learningRate": LEARNING_RATE
    },
    "featureNames": FEATURES,
    "exportTime": pd.Timestamp.now().isoformat(),
    "allTrialResults": trial_results
}

config_path = os.path.join(OUTPUT_DIR, "training_config.json")
with open(config_path, "w") as f:
    json.dump(train_config, f, indent=2)

# 打印导出信息
print(f"\n✓ 最佳模型已导出到: {model_path}")
print(f"✓ 标准化配置已导出到: {scaler_path}")
print(f"✓ 训练配置已导出到: {config_path}")
print(f"\n✓ 特征顺序: {FEATURES}")
print(f"✓ 均值 (means): {scaler.mean_.tolist()}")
print(f"✓ 标准差 (stds): {scaler.scale_.tolist()}")
print(f"\n✓ 推荐分类阈值: {best_threshold:.2f}")
print(f"✓ 最佳F1分数: {best_f1:.4f}")

# ======================
# 推理示例
# ======================
print("\n" + "="*60)
print("          推理示例")
print("="*60)

# 假设有新的输入数据
new_data = np.array([[1, 0.04, 0.04, 1, 1]])
print(f"原始输入: {new_data}")

# 使用scaler标准化
new_data_scaled = scaler.transform(new_data)
print(f"标准化后: {new_data_scaled}")

# 预测
best_model.eval()
with torch.no_grad():
    logits = best_model(torch.tensor(new_data_scaled, dtype=torch.float32))
    proba = torch.softmax(logits, dim=1)
    risk_score = proba[:, 1].item()
    print(f"风险评分: {risk_score:.4f}")
    print(f"是否风险: {risk_score >= best_threshold}")
